In [ ]:
# !pip install -q google-genai rouge-score nltk ragas
import json
import math
import nltk
import numpy as np
from google import genai
from google.genai import types as genai_types
from google.colab import userdata

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

def call_gemini(system: str, prompt: str) -> str:
    config = genai_types.GenerateContentConfig(system_instruction=system)
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text

print("Setup complete.")

Setup complete.


In [ ]:
CORPUS = {
    "doc_0": "The Eiffel Tower is located in Paris, France. It was built in 1889.",
    "doc_1": "The Louvre Museum is the world's largest art museum, also in Paris.",
    "doc_2": "The French Revolution began in 1789 and transformed French society.",
    "doc_3": "Baguettes are a staple of French cuisine, baked fresh daily.",
    "doc_4": "The Seine River flows through Paris and is 775 km long.",
    "doc_5": "Victor Hugo wrote Les Misérables, set largely in Paris.",
    "doc_6": "The Eiffel Tower attracts around 7 million visitors per year.",
    "doc_7": "French is spoken by over 300 million people worldwide.",
}

In [ ]:
QUERIES = {
    "q1": {
        "text": "When was the Eiffel Tower built and where is it?",
        "relevant_docs": {"doc_0", "doc_6"},
    },
    "q2": {
        "text": "What is the Louvre Museum?",
        "relevant_docs": {"doc_1"},
    },
    "q3": {
        "text": "Tell me about Paris landmarks",
        "relevant_docs": {"doc_0", "doc_1", "doc_4", "doc_6"},
    },
}

In [ ]:
RETRIEVED = {
    "q1": ["doc_0", "doc_3", "doc_6", "doc_2", "doc_4"],
    "q2": ["doc_3", "doc_1", "doc_5", "doc_7", "doc_2"],
    "q3": ["doc_0", "doc_6", "doc_4", "doc_7", "doc_1"],
}

In [ ]:
def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    top_k = retrieved[:k]
    hits = sum(1 for doc in top_k if doc in relevant)
    return hits / k

In [ ]:
k = 3
for qid, data in QUERIES.items():
    p = precision_at_k(RETRIEVED[qid], data["relevant_docs"], k)
    print(f"  {qid}: {p:.2f}  retrieved={RETRIEVED[qid][:k]}  relevant={data['relevant_docs']}")

  q1: 0.67  retrieved=['doc_0', 'doc_3', 'doc_6']  relevant={'doc_6', 'doc_0'}
  q2: 0.33  retrieved=['doc_3', 'doc_1', 'doc_5']  relevant={'doc_1'}
  q3: 1.00  retrieved=['doc_0', 'doc_6', 'doc_4']  relevant={'doc_1', 'doc_6', 'doc_0', 'doc_4'}


In [ ]:
def recall_at_k(retrieved: list, relevant: set, k: int) -> float:
  top_k = retrieved[:k]
  hits = sum(1 for doc in top_k if doc in relevant)
  return hits / len(retrieved)

In [ ]:
for qid, data in QUERIES.items():
  r= recall_at_k(RETRIEVED[qid], data["relevant_docs"], k)
  print(f"  {qid}: {p:.2f}  relevant_count={len(data['relevant_docs'])}")

  q1: 1.00  relevant_count=2
  q2: 1.00  relevant_count=1
  q3: 1.00  relevant_count=4


In [ ]:
def reciprocal_rank(retrieved: list,relevant: set) -> float:
  for rank,doc in enumerate(retrieved, start=1):
    if doc in relevant:
      return 1/rank
  return 0

def mean_rr(all_retrieved: dict, all_queries: dict) -> float:
  total_rr = 0
  for qid, data in all_queries.items():
    total_rr += reciprocal_rank(all_retrieved[qid], data["relevant_docs"])
  return total_rr / len(all_queries)

In [ ]:
mrr = mean_rr(RETRIEVED, QUERIES)
print(f"\nMRR = {mrr:.3f}")


MRR = 0.833


In [ ]:
# --- Summary table: all metrics across all K values ---

print(f"{'Query':<6} {'P@1':>6} {'P@3':>6} {'P@5':>6} {'R@3':>6} {'R@5':>6} {'RR':>6}")
print("-" * 44)
for qid, data in QUERIES.items():
    rel = data["relevant_docs"]
    ret = RETRIEVED[qid]
    print(
        f"{qid:<6}"
        f"{precision_at_k(ret, rel, 1):>6.2f}"
        f"{precision_at_k(ret, rel, 3):>6.2f}"
        f"{precision_at_k(ret, rel, 5):>6.2f}"
        f"{recall_at_k(ret, rel, 3):>6.2f}"
        f"{recall_at_k(ret, rel, 5):>6.2f}"
        f"{reciprocal_rank(ret, rel):>6.2f}"
    )

# STUDENT TRY: swap RETRIEVED["q1"] to ["doc_3", "doc_0", "doc_6", "doc_2", "doc_4"]
# and observe how P@1 changes even though P@3 stays the same.

Query     P@1    P@3    P@5    R@3    R@5     RR
--------------------------------------------
q1      1.00  0.67  0.40  0.40  0.40  1.00
q2      0.00  0.33  0.20  0.20  0.20  0.50
q3      1.00  1.00  0.80  0.60  0.80  1.00


In [ ]:
# --- RAG pipeline with a clean and a poisoned context ---

QUESTION = "When was the Eiffel Tower built and how tall is it?"

CLEAN_CONTEXT = """
The Eiffel Tower is located in Paris, France.
It was built in 1889 for the World's Fair.
It stands 330 metres tall including its antenna.
"""

# Poisoned context: contains a factual error planted in retrieved content
POISONED_CONTEXT = """
The Eiffel Tower is located in Paris, France.
It was built in 1923 for the Paris Olympics.
It stands 330 metres tall including its antenna.
"""

SYSTEM = "You are a helpful assistant. Answer using ONLY the provided context."

clean_answer = call_gemini(SYSTEM, f"Context:\n{CLEAN_CONTEXT}\n\nQuestion: {QUESTION}")
poisoned_answer = call_gemini(SYSTEM, f"Context:\n{POISONED_CONTEXT}\n\nQuestion: {QUESTION}")

print("=== Clean context answer ===")
print(clean_answer)
print("\n=== Poisoned context answer ===")
print(poisoned_answer)

=== Clean context answer ===
The Eiffel Tower was built in 1889 and stands 330 metres tall including its antenna.

=== Poisoned context answer ===
The Eiffel Tower was built in 1923 and stands 330 metres tall including its antenna.


In [ ]:
GROUNDING_JUDGE_SYSTEM = """You are a grounding evaluator.
Given a context, a question, and an answer, respond ONLY with valid JSON:
{"grounded": true/false, "reason": "one sentence", "unsupported_claims": ["list any claims not in context"]}
No markdown, no explanation outside the JSON."""

In [ ]:
def check_grounding(context: str, question: str, answer: str) -> dict:
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}"
    response = call_gemini(GROUNDING_JUDGE_SYSTEM, prompt)
    return json.loads(response)

In [ ]:
print(json.dumps(check_grounding(CLEAN_CONTEXT, QUESTION, clean_answer)))

{"grounded": true, "reason": "All information in the answer is directly supported by the provided context.", "unsupported_claims": []}


In [ ]:
print(POISONED_CONTEXT,QUESTION,poisoned_answer)
print(json.dumps(check_grounding(POISONED_CONTEXT, QUESTION, poisoned_answer)))


The Eiffel Tower is located in Paris, France.
It was built in 1923 for the Paris Olympics.
It stands 330 metres tall including its antenna.
 When was the Eiffel Tower built and how tall is it? The Eiffel Tower was built in 1923 and stands 330 metres tall including its antenna.
{"grounded": true, "reason": "All information in the answer is directly supported by the provided context.", "unsupported_claims": []}


In [ ]:
FAITHFULNESS_SYSTEM = """For each sentence in the answer, decide if it is fully supported
by the context. Respond ONLY with valid JSON:
{"sentences": [{"sentence": "...", "supported": true/false}]}
No markdown."""

In [ ]:
def faithfullnes_score(context: str, answer: str) -> float:
  prompt = f"Context:\n{context}\n\nAnswer:\n{answer}"
  raw = call_gemini(FAITHFULNESS_SYSTEM,prompt)
  try:
        result = json.loads(raw.strip().strip("```json").strip("```"))
        sentences = result["sentences"]
        score = sum(1 for s in sentences if s["supported"]) / len(sentences)
        for s in sentences:
            status = "✅" if s["supported"] else "❌"
            print(f"  {status} {s['sentence']}")
        return score
  except Exception:
        print("Parse failed:", raw)
        return -1.0

In [ ]:
print("=== Faithfulness: clean answer ===")
score = faithfullnes_score(CLEAN_CONTEXT, clean_answer)
print(f"  Faithfulness score: {score:.2f}\n")

print("=== Faithfulness: poisoned answer ===")
score = faithfullnes_score(CLEAN_CONTEXT, poisoned_answer)
print(f"  Faithfulness score: {score:.2f}")

=== Faithfulness: clean answer ===
  ✅ The Eiffel Tower was built in 1889 and stands 330 metres tall including its antenna.
  Faithfulness score: 1.00

=== Faithfulness: poisoned answer ===
  ❌ The Eiffel Tower was built in 1923 and stands 330 metres tall including its antenna.
  Faithfulness score: 0.00


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

REFERENCE = "The Eiffel Tower was built in 1889 and stands 330 metres tall."

candidates = {
    "exact_match":     "The Eiffel Tower was built in 1889 and stands 330 metres tall.",
    "paraphrase":      "Built in 1889, the Eiffel Tower reaches a height of 330 metres.",
    "partial":         "The Eiffel Tower was built in 1889.",
    "correct_no_ngram": "It was constructed in eighteen eighty-nine and is very tall.",
    "wrong_fact":      "The Eiffel Tower was built in 1923 and stands 330 metres tall.",
}

In [ ]:
smoother = SmoothingFunction().method1
ref_tokens = [REFERENCE.lower().split()]

for name,cand in candidates.items():
  sentence_bleu(ref_tokens, cand.lower().split(), smoothing_function=smoother)
  print(f"  {name}: {sentence_bleu(ref_tokens, cand.lower().split(), smoothing_function=smoother):.2f}")

  exact_match: 1.00
  paraphrase: 0.11
  partial: 0.40
  correct_no_ngram: 0.02
  wrong_fact: 0.73


In [ ]:
!pip install rouge-score
from rouge_score import rouge_scorer

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

print(f"{'Candidate':<20} {'R1-F':>6} {'R2-F':>6} {'RL-F':>6}")
print("-" * 40)
for name, cand in candidates.items():
    scores = scorer.score(REFERENCE, cand)
    print(
        f"{name:<20}"
        f"{scores['rouge1'].fmeasure:>6.3f}"
        f"{scores['rouge2'].fmeasure:>6.3f}"
        f"{scores['rougeL'].fmeasure:>6.3f}"
    )


Candidate              R1-F   R2-F   RL-F
----------------------------------------
exact_match          1.000 1.000 1.000
paraphrase           0.667 0.455 0.417
partial              0.737 0.706 0.737
correct_no_ngram     0.348 0.000 0.348
wrong_fact           0.917 0.818 0.917


In [ ]:
# --- Why BLEU/ROUGE are insufficient for RAG evaluation ---

print("Key limitations:\n")

limitations = [
    ("Factual errors score well",
     "wrong_fact scores high because 'stands 330 metres tall' still overlaps"),
    ("Paraphrases score low",
     "correct_no_ngram has the right meaning but zero n-gram overlap → low BLEU"),
    ("No grounding check",
     "BLEU/ROUGE compare candidate vs reference — they never check if answer is in context"),
    ("Reference dependency",
     "Requires a human-written reference answer; often unavailable in production RAG"),
    ("Order insensitivity (ROUGE-1)",
     "ROUGE-1 counts unigrams regardless of order — a jumbled sentence scores the same"),
]

for title, detail in limitations:
    print(f"  ❌ {title}")
    print(f"     → {detail}\n")

print("Use BLEU/ROUGE as a quick sanity check; use grounding/faithfulness for RAG-specific quality.")

Key limitations:

  ❌ Factual errors score well
     → wrong_fact scores high because 'stands 330 metres tall' still overlaps

  ❌ Paraphrases score low
     → correct_no_ngram has the right meaning but zero n-gram overlap → low BLEU

  ❌ No grounding check
     → BLEU/ROUGE compare candidate vs reference — they never check if answer is in context

  ❌ Reference dependency
     → Requires a human-written reference answer; often unavailable in production RAG

  ❌ Order insensitivity (ROUGE-1)
     → ROUGE-1 counts unigrams regardless of order — a jumbled sentence scores the same

Use BLEU/ROUGE as a quick sanity check; use grounding/faithfulness for RAG-specific quality.


In [ ]:
# --- Human eval dimensions for RAG outputs ---

EVAL_DIMENSIONS = {
    "Faithfulness":  "Does the answer contain only information present in the retrieved context?",
    "Relevance":     "Does the answer actually address the question asked?",
    "Completeness":  "Does the answer cover all key aspects of the question?",
    "Conciseness":   "Is the answer appropriately brief — no unnecessary padding?",
    "Fluency":       "Is the answer grammatically correct and readable?",
}

print("Human evaluation rubric (1–5 scale for each dimension):\n")
for dim, description in EVAL_DIMENSIONS.items():
    print(f"  {dim:<16} {description}")

Human evaluation rubric (1–5 scale for each dimension):

  Faithfulness     Does the answer contain only information present in the retrieved context?
  Relevance        Does the answer actually address the question asked?
  Completeness     Does the answer cover all key aspects of the question?
  Conciseness      Is the answer appropriately brief — no unnecessary padding?
  Fluency          Is the answer grammatically correct and readable?


In [ ]:
# --- Human eval dimensions for RAG outputs ---

EVAL_DIMENSIONS = {
    "Faithfulness":  "Does the answer contain only information present in the retrieved context?",
    "Relevance":     "Does the answer actually address the question asked?",
    "Completeness":  "Does the answer cover all key aspects of the question?",
    "Conciseness":   "Is the answer appropriately brief — no unnecessary padding?",
    "Fluency":       "Is the answer grammatically correct and readable?",
}

print("Human evaluation rubric (1–5 scale for each dimension):\n")
for dim, description in EVAL_DIMENSIONS.items():
    print(f"  {dim:<16} {description}")

Human evaluation rubric (1–5 scale for each dimension):

  Faithfulness     Does the answer contain only information present in the retrieved context?
  Relevance        Does the answer actually address the question asked?
  Completeness     Does the answer cover all key aspects of the question?
  Conciseness      Is the answer appropriately brief — no unnecessary padding?
  Fluency          Is the answer grammatically correct and readable?


In [ ]:
# --- Structured human eval scorecard ---

def build_eval_scorecard(question, context, answer):
    """Generates a structured scorecard for human evaluators."""
    print("=" * 60)
    print(f"QUESTION:  {question}")
    print(f"CONTEXT:   {context[:120]}...")
    print(f"ANSWER:    {answer[:200]}")
    print("=" * 60)
    print(f"\n{'Dimension':<16} {'Score (1-5)':>12}  Notes")
    print("-" * 50)
    for dim in EVAL_DIMENSIONS:
        print(f"{dim:<16} {'___':>12}  ___________________________")
    print("\nOverall score: ___ / 5")
    print("="*60)



In [ ]:
# --- End-to-end: AutoML + LLM evaluation comparison ---
# Shows how the evaluation principles from this session apply
# whether you're evaluating an AutoML model or an LLM.

import pandas as pd

print("Evaluation principles: AutoML vs LLM\n")

EVAL_COMPARISON = {
    "Dimension": [
        "Model selection",
        "Evaluation metric",
        "Overfitting risk",
        "Contamination risk",
        "Benchmark",
        "Human eval",
    ],
    "AutoML": [
        "Cross-validated accuracy on holdout",
        "Accuracy / F1 / AUC on test set",
        "CV score >> test score",
        "Train/test leakage via preprocessing on full data",
        "UCI datasets, Kaggle benchmarks",
        "Domain expert review on edge cases",
    ],
    "LLM Evaluation": [
        "Accuracy on MMLU, HellaSwag, etc.",
        "Accuracy, BLEU, ROUGE, faithfulness",
        "Benchmark-specific fine-tuning",
        "Benchmark data in pre-training corpus",
        "MMLU, HellaSwag, BIG-Bench, HELM",
        "Human raters on 5-dimension rubric",
    ],
}

df = pd.DataFrame(EVAL_COMPARISON)
print(df.to_string(index=False))

# STUDENT TRY: pick one row from this table and describe a real scenario
# where that failure mode caused a misleading evaluation result.

Evaluation principles: AutoML vs LLM

         Dimension                                            AutoML                        LLM Evaluation
   Model selection               Cross-validated accuracy on holdout     Accuracy on MMLU, HellaSwag, etc.
 Evaluation metric                   Accuracy / F1 / AUC on test set   Accuracy, BLEU, ROUGE, faithfulness
  Overfitting risk                            CV score >> test score        Benchmark-specific fine-tuning
Contamination risk Train/test leakage via preprocessing on full data Benchmark data in pre-training corpus
         Benchmark                   UCI datasets, Kaggle benchmarks      MMLU, HellaSwag, BIG-Bench, HELM
        Human eval                Domain expert review on edge cases    Human raters on 5-dimension rubric
